# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#!pip install moviepy==1.0.3
#!pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_11972\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [17]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 3 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (27 articulos)...
  Procesando: Detenida una jefa por dar trabajo a obreros extranjeros...
    -> Texto completo (1887 chars)
  Procesando: El intento de suicidio por TikTok de una niña desvela a...
    -> Texto completo (3862 chars)
  Procesando: Cierran una tienda de CBD por engañar a sus clientes co...
    -> Texto completo (3206 chars)
  Procesando: Trump busca a la desesperada una salida a la guerra en ...
    -> Texto completo (8851 chars)
  Procesando: Trump ha sido devorado por el caos que él puso en march...
    -> Texto completo (7771 chars)
  Procesando: "Soy ciudadano de EEUU e Israel ha destruido mi cas

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [18]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ABC
Título:  Detenida una jefa por dar trabajo a obreros extranjeros sin contrato por menos del salario mínimo
Origen:  completo
Chars:   1887
Preview: Valencia
La Policía Nacional ha detenido en la provincia de Valencia a dos mujeres y un hombre, de entre 37 y 55 años, por emplear a trabajadores en situación irregular en el sector de la construcción...

Artículo 2
Fuente:  ABC
Título:  El intento de suicidio por TikTok de una niña desvela a un depredador sexual en serie fantasma
Origen:  completo
Chars:   3862
Preview: La Policía Nacional ha culminado la primera parte de la operación Zalamero, la consistente en la localización y detención de un individuo de 28 años y escondido en Madrid que ha acosado virtualmente a...

Artículo 3
Fuente:  ABC
Título:  Cierran una tienda de CBD por engañar a sus clientes con cogollos, resina y porros de marihuana
Origen:  completo
Chars:   3206
Preview: Valencia
La Guardia Civil de Alicante, en el marco de la operación Senzu, ha i

## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [19]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 27 ARTÍCULOS

-> Agrupando artículos por temas...
   4 temas identificados:
   - Guerra en Oriente Próximo y su impacto (11 fuentes: ElDiario, ElDiario, Europa Press, Europa Press, Europa Press, La Vanguardia, La Vanguardia, La Vanguardia, Antena 3, RTVE, El Pais)
   - Debate sobre la Ley de Vivienda y Alquiler (4 fuentes: 20minutos, 20minutos, 20minutos, El Mundo)
   - Controversia sobre financiación de Vox (2 fuentes: El Pais, El Pais)
   - Delitos y Problemáticas Sociales (3 fuentes: ABC, ABC, ABC)

-> Generando resúmenes por tema...
   [1/4] Guerra en Oriente Próximo y su impacto (11 fuentes)...
      OK (1112 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [2/4] Debate sobre la Ley de Vivienda y Alquiler (4 fuentes)...
      OK (831 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [3/4] Controversia sobre financiación de Vox (2 fuentes)...
      OK (837 chars)
      Error: type object 'datetime.time' has no attribu

## Paso 3: Síntesis de voz (T2S)

In [20]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       4634
  Audio:       audios/audio_2026-03-24_12-23-41.mp3
Audio generado! (1.60 MB)
  Generando subtítulos con Whisper...
  Subtítulos generados: audios/subtitulos_2026-03-24_12-23-41.vtt


In [21]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 10752 chars
WEBVTT

00:00:00.000 --> 00:00:01.419
Buenas tardes y bienvenidos

00:00:01.419 --> 00:00:02.980
a nuestro boletín informativo.

00:00:03.859 --> 00:00:05.360
Hoy analizamos la escalada

00:00:05.360 --> 00:00:06.400
de tensión en Oriente

00:00:06.400 --> 00:00:07.480
próximo y su impacto

00:00:07


## Paso 4: Generación del video resumen 

In [22]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [24]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')
ruta_video = generar_video(
    ruta_audio, ruta_subtitulos, resumenes,
    PEXELS_API_KEY,
    modelo_ia=mi_modelo_gemini  # para mejorar las queries de imagen
)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    278.8s
   Temas:             4
   Duración por tema: 69.7s

2. Buscando imágenes en Pexels...
  Tema: Guerra en Oriente Próximo y su impacto...
    Query imagen: 'War torn Middle East'
    Keywords: 'torn middle east'
    Imagen guardada: imagenes/img_684909901344989330.jpg
  Tema: Debate sobre la Ley de Vivienda y Alquiler...
    Query imagen: 'Housing Crisis Protest'
    Keywords: 'housing crisis protest'
    Imagen guardada: imagenes/img_4555779985746359575.jpg
  Tema: Controversia sobre financiación de Vox...
    Query imagen: 'Dark Money Politics Scandal'
    Keywords: 'dark money politics scandal'
    Imagen guardada: imagenes/img_5135775646035811042.jpg
  Tema: Delitos y Problemáticas Sociales...
    Keywords: 'delitos problemáticas sociales'
    Imagen guardada: imagenes/img_80751536125904159.jpg

3. Creando clips...
   [1/4] Guerra en Oriente Próximo y su impacto...
   [2/4] Debate sobre la Ley de Vivienda y Alquil..